# 🔗 SOTA Sequential Classifier Chain Stacking & Calibration (Historical Re-creation)
### Match: NVIDIA Multi-Regressor Stacker v13 (High Variance S-targets) (Score: 0.59642)

This notebook re-creates the sequential classifier chain pipeline.
It chains target predictions step-by-step to preserve cross-target physiological interactions while strictly avoiding look-ahead leakage.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import log_loss
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

project_root = Path(os.getcwd()).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import DataLoader
from src.data.make_dataset import build_training_dataset
from src.features.build_features import engineer_features
from src.features.transformers import LeakFreeSubjectZScaler, FoldWisePCA, LeakFreeSubjectTargetEncoder
from src.utils.validation import make_subject_hole_folds

print("✅ Executable SOTA Classifier Chain template successfully imported!")

In [ ]:
data_loader = DataLoader(data_dir="data")
metrics_df = data_loader.load_metrics()
sample_sub_df = data_loader.load_sample_submission()

processed_train_path = Path("data/processed/df_train_raw.parquet")
if processed_train_path.exists():
    df_train_raw = pd.read_parquet(processed_train_path)
else:
    sensor_data_dict = data_loader.load_sensor_data()
    df_train_raw = build_training_dataset(metrics_df, sensor_data_dict)

df_train_raw['is_test'] = 0
df_train = engineer_features(df_train_raw)
df_train = df_train.sort_values(by=['subject_id', 'date']).reset_index(drop=True)
print(f"Loaded Train Dataset shape: {df_train.shape}")